# 01 — Files et `pathlib`

## Learning Objectives

By the end of this notebook, you will know how to :

- lire et écrire un file texte avec `open` et `with` ;
- usesr les **modes** (`'r'`, `'w'`, `'a'`) ;
- comprendre pourquoi **toujours** spécifier l'encodage (`encoding='utf-8'`) ;
- usesr `Path.read_text` et `Path.write_text` for thes cas simples ;
- itérer ligne par ligne sans charger tout le file.

## Prérequis

- all the notions previouses ;
- `pathlib`, exceptions.

Pas encore vus :

- CSV (notebook following), JSON.

## Outline

1. `open` et le contexte `with`
2. Modes de file
3. Encodage explicite
4. Lecture : `read`, `readlines`, iteration
5. Écriture : `write`, `writelines`
6. `Path.read_text` / `Path.write_text`
7. Gestion d'erreurs classiques
8. Synthèse
9. Exercises

---


## 1. `open` et le contexte `with`

`open(chemin, mode)` ouvre un file et returns un objet file. On l'uses **toujours** dans un `with` pour garantir la closure automatique.

In [ ]:
from pathlib import Path
path = Path('/tmp/exemple.txt')

with open(path, 'w', encoding='utf-8') as f:
    f.write('Bonjour, monde !\n')
    f.write('Deuxième line.\n')

In [ ]:
with open(path, 'r', encoding='utf-8') as f:
    content = f.read()
print(content)

---


## 2. Modes de file

| Mode | Effet | Si exists | Si absent |
|---|---|---|---|
| `'r'` | lecture (by default) | lit | `FileNotFoundError` |
| `'w'` | écriture | **écrase** | creates |
| `'a'` | ajout | adds to the fin | creates |
| `'x'` | création exclusive | `FileExistsError` | creates |
| `'rb'` / `'wb'` | binaire | idem | idem |

**Pour du texte, ne pas mettre de `b`.**

In [ ]:
from pathlib import Path
path = Path('/tmp/exemple.txt')

with open(path, 'a', encoding='utf-8') as f:
    f.write('Ligne ajoutée.\n')

print(path.read_text(encoding='utf-8'))

---


## 3. Encodage explicite

**Toujours** spécifier `encoding='utf-8'`. Sur Windows, l'encodage by default dépend of the locale — ce qui creates des bugs invisibles on thes others machines.

### PEP 597 : avertissement si encoding n'est pas spécifié

Since Python 3.10, Python **recommande** d'écrire explicitement `encoding=...`. En 3.15 ça deviendra une erreur.

---


## 4. Lecture

### `read()` — tout le file en une chaîne

In [ ]:
from pathlib import Path
path = Path('/tmp/exemple.txt')

with open(path, 'r', encoding='utf-8') as f:
    content = f.read()

print(len(content), 'characters')

### `readlines()` — list de lignes (avec `\n` to the fin)

In [ ]:
with open(path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(lines)

### Iteration directe — idiomatique et économe

In [ ]:
with open(path, 'r', encoding='utf-8') as f:
    for line in f:
        print(repr(line))

L'iteration directe **ne charge pas tout le file** en mémoire. Préférer cette forme for thes gros files.

---


## 5. Écriture

In [ ]:
lines = ['un\n', 'deux\n', 'trois\n']
with open('/tmp/numbers.txt', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print(open('/tmp/numbers.txt', encoding='utf-8').read())

⚠️ `writelines` n'adds **pas** de `\n` automatiquement. C'est à vous de le faire.

---


## 6. `Path.read_text` / `Path.write_text`

Pour les cas simples (tout lire, tout écrire), `pathlib` offre des raccourcis plus lisibles.

In [ ]:
from pathlib import Path
p = Path('/tmp/rapide.txt')
p.write_text('Hello world\n', encoding='utf-8')

In [ ]:
p.read_text(encoding='utf-8')

⚠️ `write_text` **écrase** le file. Pour addsr, il faut passer par `open('a')`.

---


## 7. Gestion d'erreurs

In [ ]:
try:
    with open('/inexistant.txt', 'r', encoding='utf-8') as f:
        f.read()
except FileNotFoundError as err:
    print('file introuvable :', err)

In [ ]:
try:
    with open('/root/protege.txt', 'w', encoding='utf-8') as f:
        f.write('test')
except PermissionError as err:
    print('droits insuffisants :', err)

---


## 8. Synthèse

| Besoin | Forme |
|---|---|
| Ouvrir en lecture | `open(p, 'r', encoding='utf-8')` |
| Lire tout | `f.read()` ou `Path.read_text()` |
| Itérer ligne par ligne | `for ligne in f:` |
| Écrire (écraser) | `open(p, 'w', ...)` ou `Path.write_text()` |
| Ajouter to the fin | `open(p, 'a', ...)` |
| Créer sans écraser | `open(p, 'x', ...)` |

### Règles

1. Toujours `with`.
2. Toujours `encoding='utf-8'`.
3. Itérer directement for thes gros files.
4. `Path.read_text` / `write_text` pour the code concis.
5. Attraper `FileNotFoundError`, `PermissionError`, `IsADirectoryError` selon le contexte.

---


## 9. Exercises

### Exercise 1 — Écrire et relire *(facile)*

Écrire `sauver_et_relire(contenu: str, chemin: str) -> str` qui écrit `contenu` dans `chemin` puis le relit et le returns.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
from pathlib import Path

def sauver_et_relire(contenu: str, chemin: str) -> str:
    """Écrit puis relit le contenu."""
    p = Path(chemin)
    p.write_text(contenu, encoding='utf-8')
    return p.read_text(encoding='utf-8')

print(sauver_et_relire('hello', '/tmp/demo_sauver.txt'))
```

</details>

### Exercise 2 — Compter les lignes *(facile)*

Écrire `compter_lignes(chemin: str) -> int` which returns le nombre de lignes du file.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def compter_lignes(chemin: str) -> int:
    """Nombre de lignes d'un file texte."""
    total = 0
    with open(chemin, 'r', encoding='utf-8') as f:
        for _ in f:
            total = total + 1
    return total

# préparer un file pour tester
from pathlib import Path
Path('/tmp/demo_lignes.txt').write_text('un\ndeux\ntrois\n', encoding='utf-8')
print(compter_lignes('/tmp/demo_lignes.txt'))
```

</details>

### Exercise 3 — Ajouter une ligne *(facile)*

Écrire `addsr_ligne(chemin: str, ligne: str) -> None` qui adds `ligne` to the fin du file (avec `\n`). Utiliser le mode `'a'`.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def addsr_ligne(chemin: str, ligne: str) -> None:
    """Ajoute une ligne to the fin du file."""
    with open(chemin, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')

from pathlib import Path
Path('/tmp/demo_append.txt').write_text('initial\n', encoding='utf-8')
addsr_ligne('/tmp/demo_append.txt', 'ajouté')
print(Path('/tmp/demo_append.txt').read_text(encoding='utf-8'))
```

</details>

### Exercise 4 — Lecture tolérante *(moyen)*

Écrire `lire_ou_defaut(chemin: str, defaut: str) -> str` which returns le contenu du file, ou `defaut` si le file n'exists pas.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def lire_ou_defaut(chemin: str, defaut: str) -> str:
    """Lit le file ou returns defaut si absent."""
    try:
        with open(chemin, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return defaut

print(lire_ou_defaut('/absent.txt', 'VIDE'))
```

</details>

### Exercise 5 — Inverser les lignes *(moyen)*

Écrire `reversesr_lignes(chemin: str) -> None` qui lit un file et le réécrit with thes lignes dans l'ordre reverses.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def reversesr_lignes(chemin: str) -> None:
    """Réécrit le file en inversant l'ordre des lignes."""
    with open(chemin, 'r', encoding='utf-8') as f:
        lignes = f.readlines()
    with open(chemin, 'w', encoding='utf-8') as f:
        f.writelines(reversed(lignes))

from pathlib import Path
Path('/tmp/demo_inv.txt').write_text('a\nb\nc\n', encoding='utf-8')
reversesr_lignes('/tmp/demo_inv.txt')
print(Path('/tmp/demo_inv.txt').read_text(encoding='utf-8'))
```

</details>

### Exercise 6 — Compter les mots *(moyen)*

Écrire `compter_mots(chemin: str) -> dict[str, int]` which returns un dict `{mot: nombre}` pour tout le file (découper avec `.split()`).

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def compter_mots(chemin: str) -> dict[str, int]:
    """Compte les mots d'un file texte."""
    freq: dict[str, int] = {}
    with open(chemin, 'r', encoding='utf-8') as f:
        for ligne in f:
            for mot in ligne.split():
                freq[mot] = freq.get(mot, 0) + 1
    return freq

from pathlib import Path
Path('/tmp/demo_mots.txt').write_text('un deux un\ntrois un deux\n', encoding='utf-8')
print(compter_mots('/tmp/demo_mots.txt'))
```

</details>

### Exercise 7 — Filtrer un file *(difficile)*

Écrire `filtrer(source: str, destination: str, motif: str) -> int` qui copies `source` vers `destination` en ne gardant que les lignes **contenant** `motif`. Renvoyer le nombre de lignes copiées.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def filtrer(source: str, destination: str, motif: str) -> int:
    """Copie les lignes contenant `motif`."""
    n = 0
    with open(source, 'r', encoding='utf-8') as betweene:
        with open(destination, 'w', encoding='utf-8') as sortie:
            for ligne in betweene:
                if motif in ligne:
                    sortie.write(ligne)
                    n = n + 1
    return n

from pathlib import Path
Path('/tmp/demo_src.txt').write_text('abc\nxyz\nabx\n', encoding='utf-8')
print(filtrer('/tmp/demo_src.txt', '/tmp/demo_dst.txt', 'ab'))
print(Path('/tmp/demo_dst.txt').read_text(encoding='utf-8'))
```

</details>

### Exercise 8 — Concaténer several files *(difficile)*

Écrire `concatener(chemins: list[str], destination: str) -> int` qui concatène all the files `chemins` dans `destination`. Renvoyer le nombre total de becauseactères écrits.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def concatener(chemins: list[str], destination: str) -> int:
    """Concatène several files en un seul."""
    total = 0
    with open(destination, 'w', encoding='utf-8') as sortie:
        for chemin in chemins:
            with open(chemin, 'r', encoding='utf-8') as betweene:
                contenu = betweene.read()
                sortie.write(contenu)
                total = total + len(contenu)
    return total

from pathlib import Path
Path('/tmp/a.txt').write_text('un\n', encoding='utf-8')
Path('/tmp/b.txt').write_text('deux\n', encoding='utf-8')
print(concatener(['/tmp/a.txt', '/tmp/b.txt'], '/tmp/ab.txt'))
```

</details>

---


## Ressources externes

- [Lecture/écriture — tutoriel](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)
- [`open()` — doc](https://docs.python.org/3/library/functions.html#open)
- [`pathlib`](https://docs.python.org/3/library/pathlib.html)
- [PEP 597 — `encoding` explicite](https://peps.python.org/pep-0597/)

---

## Mini-Examples supplémentaires

### Vérifier l'existsnce

In [ ]:
from pathlib import Path
print(Path('/tmp').exists())
print(Path('/inexistant').exists())

### Créer un dossier

In [ ]:
from pathlib import Path
p = Path('/tmp/demo_dir')
p.mkdir(exist_ok=True)
print(p.is_dir())

### `Path.glob` pour listr selon un motif

In [ ]:
from pathlib import Path
# List les .py du dossier /tmp/demo_dir (s'il exists)
for p in Path('/tmp').glob('*.txt'):
    print(p.name)

### Taille d'un file

In [ ]:
from pathlib import Path
p = Path('/tmp/demo_taille.txt')
p.write_text('hello world\n', encoding='utf-8')
print(p.stat().st_size, 'octets')

### Chemin absolu / résolu

In [ ]:
from pathlib import Path
p = Path('./path/../relatif')
print(p)
print(p.resolve())

### Renommer, déplacer

In [ ]:
from pathlib import Path
src = Path('/tmp/a_renommer.txt')
src.write_text('hi', encoding='utf-8')
dest = Path('/tmp/renomme.txt')
src.replace(dest)
print(dest.read_text(encoding='utf-8'))

### Supprimer

In [ ]:
from pathlib import Path
p = Path('/tmp/a_supprimer.txt')
p.write_text('bye', encoding='utf-8')
p.unlink()
print(p.exists())

### Binaire

In [ ]:
from pathlib import Path
p = Path('/tmp/demo.bin')
p.write_bytes(b'\x00\x01\x02\x03')
print(p.read_bytes())

### Encodages others que UTF-8

In [ ]:
from pathlib import Path
p = Path('/tmp/latin1.txt')
p.write_text('Éléphant', encoding='latin-1')
# Si on relit en utf-8, on risque un UnicodeDecodeError
print(p.read_text(encoding='latin-1'))